# logit-thermo 探索 Y の続き 2：タイトルは「前提」を担っているか、離れた文どうしのつながりはあるか（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは 20〜30 分。
- **探索**です。確かめには使いません。生成はせず、探索 Y と同じ BBC の記事の窓（38 本）を 4 つのモデルに読ませるだけです。
- タイトルの 3 条件（なし・その記事のタイトル・別の記事のタイトル）× 並べ替え 7 つ（探索 Y の 5 つ ＋「最初の文だけ残して残りを逆順」「最初の段落だけ残して残りの段落を逆順」）で、本文の確率を測ります。
- 読み方（R1〜R4、参考 R5）は回す前に決めて、`scripts/a2_explore_title.py` の説明文に書いてあります。
- タイトルも本文も、中身は表示も保存もしません。保存先 `A2_explore/title_summary.json` と `A2_explore/title_{モデル}.npz`（数字だけ）。
- 計算のコード：`scripts/a2_2_collect.py`（`e578cfc480980be6…`）、`scripts/a2_explore_xy.py`（`5c5750e76358b86a…`）、`scripts/a2_explore_title.py`（`62cd36ef91212dba…`）を使います（はじめの 2 つは、Drive にある探索 X・Y のノートブックから取り出して、ハッシュで確かめます。**`A2_explore_xy_L4.ipynb` を logit-thermo フォルダから動かさないでください**）。
- **通知**：🚀・モデルごとの ✅・🎉・⚠️。名札は「logit-thermo 探索 Y の続き 2（タイトルは前提を担うか・離れた文のつながり、4 モデル、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 Y の続き 2（タイトルは前提を担うか・離れた文のつながり、4 モデル、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore title', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 14e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU が足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
# a2_2_collect.py と a2_explore_xy.py は、Drive にある探索 X・Y のノートブック（A2_explore_xy_L4.ipynb）に埋め込んだものを
# 取り出して使う（ハッシュで中身が同じことを確かめる）。このノートブックに埋め込むのは a2_explore_title.py だけ。
import ast, glob, hashlib, json
_cands = sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/A2_explore_xy_L4*.ipynb', recursive=True))
assert _cands, 'Drive の logit-thermo フォルダに A2_explore_xy_L4.ipynb が見つかりません'
_got = {}
for _p in _cands:
    for _c in json.load(open(_p))['cells']:
        _s = _c['source'] if isinstance(_c['source'], str) else ''.join(_c['source'])
        if 'SRC22 = ' not in _s:
            continue
        for _n in ast.parse(_s).body:
            if isinstance(_n, ast.Assign) and isinstance(_n.targets[0], ast.Name) and _n.targets[0].id in ('SRC22', 'SRCXY'):
                _got[_n.targets[0].id] = ast.literal_eval(_n.value)
    if len(_got) == 2:
        break
SRC22, SRCXY = _got['SRC22'], _got['SRCXY']
SRCT = '"""探索 Y の続き 2：タイトルは「前提」を担っているか、離れた文どうしのつながりはあるか（2026-10-28 夜）。確かめには使わない。\nGPU（L4）のノートブックで、scripts/a2_2_collect.py と scripts/a2_explore_xy.py を実行したのと同じ名前空間で実行する\n（load_docs・load_model・valid_mask・y_window・y_orders・BBC_MONTHS を使う）。生成はしない（人の文章を読むだけ）。\n\n材料：探索 X・Y と同じ BBC の 40 本の窓（38 本）、同じ 4 モデル。\nタイトルの 3 条件（本文の前に置く書き出し。書き出しと本文は別々にトークンに区切ってつなぐ。測るのは本文のトークンの log p の合計）：\n  - none：「Article:\\\\n」（探索 Y と同じ）\n  - match：「Title: <その記事のタイトル>\\\\nArticle:\\\\n」\n  - mismatch：「Title: <別の記事のタイトル>\\\\nArticle:\\\\n」（38 本の中で 1 本ずつずらす。タイトルの書き方は同じ BBC のまま、中身だけ合わない）\n並べ替え 7 つ（文の番号の並び）：\n  - original、sent_rev、para_rev、shuf1、shuf2：探索 Y と同じ。\n  - lead_rev：最初の文だけ先頭に残し、残りの文を逆順にする（壊れる隣の組の数は sent_rev と同じ「文の数 − 1」）。\n  - para_lead：最初の段落だけ先頭に残し、残りの段落を逆順にする（段落の中の順番はそのまま。段落が 3 つ以上の記事だけ）。\n各並べ替えについて数えるもの：\n  - b：壊れた隣の組の数（元で隣り合っていた (i, i+1) が、同じ向きで隣に残らなかった数）。\n  - K_long：逆向きになった「離れた組」の数（i < j かつ j − i ≥ 2 で、j が i より前に来た組の数）。\n  - lead：最初の文が先頭にいないなら 1、いるなら 0。\n戻れなさ σ ＝ log p（original）− log p（並べ替え）（同じタイトル条件の中で引く）。\n\n読み方（回す前に決める。探索の目安。95% 区間は記事を選び直すブートストラップ 2,000 回、種 20261031）：\n  R1 合うタイトルは書き出しの損を減らすか：書き出しの損 L ＝ σ(sent_rev) − σ(lead_rev)（記事ごと。最初の文を先頭からどかした損で、\n     最初の文と、離れた後ろの文とのつながりの分も含む）。\n     L(none) − L(match) の平均の 95% 区間が 4 モデルとも 0 より上 →「合うタイトルで書き出しの損が減る」\n     4 モデルとも 0 を含むか下 →「合うタイトルでも書き出しの損は減らない」\u3000それ以外 →「はっきりしない」\n  R2 無関係なタイトルはどう効くか：L(mismatch) − L(none) の平均の 95% 区間で、\n     4 モデルとも 0 を含む →「無関係なタイトルは書き出しの損を変えない」\n     4 モデルとも 0 より下 →「無関係なタイトルでも書き出しの損が減る（タイトルがあるという形だけで効く）」\n     4 モデルとも 0 より上 →「無関係なタイトルで書き出しの損が増える」\u3000それ以外 →「はっきりしない」\n  R3 書き出しを残しても、段落の境目は高いか（none）：記事ごとの [σ(para_lead) ÷ 段落の数 − 1] ÷ [σ(lead_rev) ÷ 文の数 − 1]\n     の中央値（段落が 3 つ以上の記事）。前回と同じ線引き：4 モデルとも 0.67〜1.5 →「書き出しを残すと、境目の数でだいたい説明できる」、\n     4 モデルとも 1.5 より大きい →「書き出しを残しても、段落の境目は高い」、4 モデルとも 0.67 未満 →「書き出しを残すと、段落の境目は安い」、\n     それ以外 →「はっきりしない」\n  R4 離れた文どうしのつながりはあるか（none）：6 つの並べ替えをまとめて σ ＝ A × lead ＋ c_adj × b ＋ c_long × K_long を当てはめる\n     （切片なし）。c_long の 95% 区間が 4 モデルとも 0 より上 →「離れたつながりがある」、4 モデルとも 0 を含む →「離れたつながりは見えない」、\n     それ以外 →「はっきりしない」。A・c_adj も記録する。\n  参考 R5：前回の当てはめ σ ＝ a ＋ c × b（none、sent_rev・para_rev・shuf1・shuf2）の a の 95% 区間。OLMo の a（前回ほぼ 0）が、\n     ほかの 3 モデルの a の点の値を区間に含めば「OLMo の a はぶれで説明できる」。\n  参考：none の log p が探索 Y の値（xy_{モデル}.npz）とどれだけずれるか（同じ計算の再現の確認。記録だけ）。\nタイトルの中身は表示しない・保存しない（トークンの数だけ記録する）。\n"""\nimport gc, json, math, time\nimport numpy as np\nimport torch\n\nT_MODELS = [(\'pythia-1b\', \'EleutherAI/pythia-1b\'), (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\'),\n            (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\'), (\'smollm2-1.7b\', \'HuggingFaceTB/SmolLM2-1.7B\')]\nT_CONDS = [\'none\', \'match\', \'mismatch\']\nT_ORDERS = [\'original\', \'sent_rev\', \'para_rev\', \'shuf1\', \'shuf2\', \'lead_rev\', \'para_lead\']\nT_BOOT, T_BOOT_SEED = 2000, 20261031\nT_LOW, T_HIGH = 0.67, 1.5\n\n\ndef load_titles(docs):\n    from datasets import load_dataset\n    want = {d[\'link\'] for d in docs}\n    got = {}\n    for m in BBC_MONTHS:\n        ds = load_dataset(\'RealTimeData/bbc_news_alltime\', m, split=\'train\')\n        assert \'title\' in ds.column_names, ds.column_names\n        for row in ds:\n            if row[\'link\'] in want and row[\'link\'] not in got:\n                got[row[\'link\']] = (row[\'title\'] or \'\').strip()\n    titles = [got.get(d[\'link\'], \'\') for d in docs]\n    assert all(titles), \'タイトルが見つからない記事があります\'\n    return titles\n\n\ndef order_indices(win):\n    """段落ごとの文のリスト → 並べ替えごとの文の番号の並び（para_lead は段落が 3 つ未満なら None）。"""\n    sizes = [len(p) for p in win]\n    ns = sum(sizes)\n    starts = np.cumsum([0] + sizes[:-1])\n    paras = [list(range(s, s + n)) for s, n in zip(starts, sizes)]\n    out = {\'original\': list(range(ns)), \'sent_rev\': list(range(ns))[::-1],\n           \'para_rev\': [i for p in paras[::-1] for i in p]}\n    for k in [1, 2]:\n        out[f\'shuf{k}\'] = [int(i) for i in np.random.default_rng(k).permutation(ns)]   # y_orders と同じ作り方\n    out[\'lead_rev\'] = [0] + list(range(1, ns))[::-1]\n    out[\'para_lead\'] = paras[0] + [i for p in paras[1:][::-1] for i in p] if len(paras) >= 3 else None\n    return out\n\n\ndef order_stats(perm):\n    """文の番号の並び → (b、K_long、lead)。"""\n    pos = np.empty(len(perm), int); pos[np.asarray(perm)] = np.arange(len(perm))\n    n = len(perm)\n    b = int(sum(1 for i in range(n - 1) if pos[i + 1] != pos[i] + 1))\n    K_long = int(sum(1 for i in range(n) for j in range(i + 2, n) if pos[j] < pos[i]))\n    return b, K_long, int(perm[0] != 0)\n\n\n@torch.no_grad()\ndef body_logprob(model, tok, prefix, text, valid, device):\n    pre = tok(prefix, add_special_tokens=False).input_ids\n    bos = [tok.bos_token_id] if (tok.bos_token_id is not None and getattr(tok, \'add_bos_token\', False)) else []\n    ids = pre + tok(text, add_special_tokens=False).input_ids\n    x = torch.tensor([bos + ids], device=device)\n    z = model(x).logits[0].double().masked_fill(~valid, float(\'-inf\'))\n    lp = torch.log_softmax(z, -1)\n    start = len(bos) + len(pre)\n    y = x[0, start:]\n    tgt = lp[start - 1:x.shape[1] - 1].gather(-1, y[:, None])[:, 0]\n    return float(tgt.sum()), int(len(y)), int(len(pre))\n\n\ndef prefixes(titles, j):\n    n = len(titles)\n    return {\'none\': \'Article:\\n\', \'match\': f\'Title: {titles[j]}\\nArticle:\\n\', \'mismatch\': f\'Title: {titles[(j + 1) % n]}\\nArticle:\\n\'}\n\n\ndef boot_ci(v, rng_seed=T_BOOT_SEED, B=T_BOOT):\n    v = np.asarray(v, float)\n    rng = np.random.default_rng(rng_seed)\n    idx = rng.integers(0, len(v), size=(B, len(v)))\n    m = v[idx].mean(1)\n    return [float(v.mean()), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))]\n\n\ndef fit_lsq(rows, cols):\n    """rows：記事ごとの [(σ, 説明変数...)] のリスト。cols：使う説明変数の数。→ 係数（切片なし）。"""\n    X = np.array([r[1:1 + cols] for doc in rows for r in doc], float)\n    y = np.array([r[0] for doc in rows for r in doc], float)\n    return np.linalg.lstsq(X, y, rcond=None)[0]\n\n\ndef boot_fit(rows, cols, B=T_BOOT, seed=T_BOOT_SEED):\n    est = fit_lsq(rows, cols)\n    rng = np.random.default_rng(seed)\n    bs = np.array([fit_lsq([rows[i] for i in rng.integers(0, len(rows), len(rows))], cols) for _ in range(B)])\n    return [[float(est[k]), float(np.percentile(bs[:, k], 2.5)), float(np.percentile(bs[:, k], 97.5))] for k in range(cols)]\n\n\ndef ci_word(cis, pos, zero, neg, other=\'はっきりしない\'):\n    if all(lo > 0 for _, lo, _ in cis):\n        return pos\n    if all(lo <= 0 <= hi for _, lo, hi in cis):\n        return zero\n    if neg is not None and all(hi < 0 for _, _, hi in cis):\n        return neg\n    return other\n\n\ndef ratio_word(meds):\n    if all(T_LOW <= v <= T_HIGH for v in meds):\n        return \'書き出しを残すと、境目の数でだいたい説明できる\'\n    if all(v > T_HIGH for v in meds):\n        return \'書き出しを残しても、段落の境目は高い\'\n    if all(v < T_LOW for v in meds):\n        return \'書き出しを残すと、段落の境目は安い\'\n    return \'はっきりしない\'\n\n\ndef analyze_title(LP, wins):\n    """LP：{モデル: (記事, 3 条件, 7 並べ替え) の log p（para_lead がない記事は nan）}。"""\n    idx = [order_indices(w) for w in wins]\n    st = [{k: (order_stats(v) if v is not None else None) for k, v in o.items()} for o in idx]\n    ns = np.array([len(o[\'original\']) for o in idx]); npar = np.array([len(w) for w in wins])\n    has_pl = npar >= 3\n    oi = {k: i for i, k in enumerate(T_ORDERS)}\n    res = {\'n_docs\': len(wins), \'n_docs_para_lead\': int(has_pl.sum()), \'models\': {}}\n    for m, lp in LP.items():\n        sig = lp[:, :, [0]] - lp                                    # (記事, 条件, 並べ替え)\n        r = {}\n        L = {c: sig[:, ci, oi[\'sent_rev\']] - sig[:, ci, oi[\'lead_rev\']] for ci, c in enumerate(T_CONDS)}\n        r[\'lead_cost\'] = {c: boot_ci(L[c]) for c in T_CONDS}\n        r[\'lead_cost_none_minus_match\'] = boot_ci(L[\'none\'] - L[\'match\'])\n        r[\'lead_cost_mismatch_minus_none\'] = boot_ci(L[\'mismatch\'] - L[\'none\'])\n        r[\'sigma_mean\'] = {c: {k: float(np.nanmean(sig[:, ci, oi[k]])) for k in T_ORDERS[1:]} for ci, c in enumerate(T_CONDS)}\n        cp = sig[has_pl, 0, oi[\'para_lead\']] / (npar[has_pl] - 1)\n        cs = sig[has_pl, 0, oi[\'lead_rev\']] / (ns[has_pl] - 1)\n        r[\'lead_kept_para_over_sent_quartiles\'] = [float(np.percentile(cp / cs, q)) for q in (25, 50, 75)]\n        rows = []\n        for j in range(len(wins)):\n            doc = []\n            for k in T_ORDERS[1:]:\n                if st[j][k] is None:\n                    continue\n                b, K, ld = st[j][k]\n                doc.append((sig[j, 0, oi[k]], ld, b, K))\n            rows.append(doc)\n        A, c_adj, c_long = boot_fit(rows, 3)\n        r[\'fit_lead_adj_long\'] = {\'A\': A, \'c_adj\': c_adj, \'c_long\': c_long}\n        rows5 = [[(sig[j, 0, oi[k]], 1.0, st[j][k][0]) for k in [\'sent_rev\', \'para_rev\', \'shuf1\', \'shuf2\']] for j in range(len(wins))]\n        a5, c5 = boot_fit(rows5, 2)\n        r[\'fit_a_plus_cb\'] = {\'a\': a5, \'c\': c5}\n        res[\'models\'][m] = r\n    M = list(LP)\n    res[\'R1\'] = ci_word([res[\'models\'][m][\'lead_cost_none_minus_match\'] for m in M],\n                        \'合うタイトルで書き出しの損が減る\', \'合うタイトルでも書き出しの損は減らない\', \'合うタイトルでも書き出しの損は減らない\')\n    res[\'R2\'] = ci_word([res[\'models\'][m][\'lead_cost_mismatch_minus_none\'] for m in M],\n                        \'無関係なタイトルで書き出しの損が増える\', \'無関係なタイトルは書き出しの損を変えない\',\n                        \'無関係なタイトルでも書き出しの損が減る（タイトルがあるという形だけで効く）\')\n    res[\'R3\'] = ratio_word([res[\'models\'][m][\'lead_kept_para_over_sent_quartiles\'][1] for m in M])\n    res[\'R4\'] = ci_word([res[\'models\'][m][\'fit_lead_adj_long\'][\'c_long\'] for m in M],\n                        \'離れたつながりがある\', \'離れたつながりは見えない\', None)\n    if \'olmo2-1b\' in LP:\n        lo, hi = res[\'models\'][\'olmo2-1b\'][\'fit_a_plus_cb\'][\'a\'][1:]\n        others = [res[\'models\'][m][\'fit_a_plus_cb\'][\'a\'][0] for m in M if m != \'olmo2-1b\']\n        res[\'R5\'] = \'OLMo の a はぶれで説明できる\' if all(lo <= v <= hi for v in others) else \'OLMo の a はぶれだけでは説明できない\'\n    res[\'counts\'] = {\'ns\': ns.tolist(), \'np\': npar.tolist(), \'para_sizes\': [[len(p) for p in w] for w in wins],\n                     \'stats\': [{k: v for k, v in s.items()} for s in st]}\n    return res\n\n\ndef run_title(docs, outdir, device, notify=print):\n    wins_all = [y_window(d[\'text\']) for d in docs]\n    keep = [i for i, w in enumerate(wins_all) if w is not None]\n    wins = [wins_all[i] for i in keep]\n    titles_all = load_titles(docs)\n    titles = [titles_all[i] for i in keep]\n    assert len(set(titles)) == len(titles), \'タイトルが重なっています（ずらしても合ってしまう）\'\n    LP, info = {}, {}\n    for name, hf_id in T_MODELS:\n        t0 = time.time()\n        tok, model = load_model(hf_id, device)\n        valid, _ = valid_mask(tok, model, device)\n        lp = np.full((len(wins), len(T_CONDS), len(T_ORDERS)), np.nan)\n        nt = np.zeros_like(lp); npre = np.zeros((len(wins), len(T_CONDS)))\n        for j, w in enumerate(wins):\n            sents = [s for p in w for s in p]\n            oi = order_indices(w)\n            yo = y_orders(w)\n            for k in [\'original\', \'sent_rev\', \'para_rev\', \'shuf1\', \'shuf2\']:\n                assert \' \'.join(sents[i] for i in oi[k]) == yo[k], (\'並べ替えが探索 Y と違います\', k)\n            pre = prefixes(titles, j)\n            for ci, c in enumerate(T_CONDS):\n                for ki, k in enumerate(T_ORDERS):\n                    if oi[k] is None:\n                        continue\n                    lp[j, ci, ki], nt[j, ci, ki], npre[j, ci] = body_logprob(model, tok, pre[c], \' \'.join(sents[i] for i in oi[k]), valid, device)\n        try:\n            old = np.load(f\'{outdir}/xy_{name}.npz\')[\'y_logp\']\n            info[name] = {\'max_abs_diff_vs_xy_none\': float(np.max(np.abs(old - lp[:, 0, :5])))}\n        except Exception as e:\n            info[name] = {\'max_abs_diff_vs_xy_none\': None, \'note\': type(e).__name__}\n        info[name][\'title_tokens_mean\'] = float(npre[:, 1].mean() - npre[:, 0].mean())\n        info[name][\'seconds\'] = round(time.time() - t0, 1)\n        np.savez_compressed(f\'{outdir}/title_{name}.npz\', logp=lp, ntok=nt, npre=npre, conds=np.array(T_CONDS), orders=np.array(T_ORDERS))\n        LP[name] = lp\n        del model; gc.collect(); torch.cuda.empty_cache()\n        notify(f\'{name} おわり（{time.time() - t0:.0f} 秒）\')\n    res = analyze_title(LP, wins)\n    res[\'run_info\'] = info\n    return res\n\n\n@torch.no_grad()\ndef self_test_title(device):\n    # 並べ替えと数え方\n    win = [[\'s0 aa.\', \'s1 aa.\'], [\'s2 aa.\', \'s3 aa.\', \'s4 aa.\'], [\'s5 aa.\'], [\'s6 aa.\', \'s7 aa.\']]\n    o = order_indices(win)\n    assert o[\'lead_rev\'] == [0, 7, 6, 5, 4, 3, 2, 1] and o[\'para_lead\'] == [0, 1, 6, 7, 5, 2, 3, 4], o\n    assert order_stats(o[\'original\']) == (0, 0, 0)\n    assert order_stats(o[\'sent_rev\']) == (7, 21, 1)                  # 8 文：隣 7 組すべて、離れた組 28 − 7 ＝ 21\n    assert order_stats(o[\'lead_rev\'])[0] == 7 and order_stats(o[\'lead_rev\'])[2] == 0\n    b, K, ld = order_stats(o[\'para_rev\'])\n    assert b == 3 and ld == 1, (b, K, ld)\n    sents = [s for p in win for s in p]; yo = y_orders(win)\n    for k in [\'original\', \'sent_rev\', \'para_rev\', \'shuf1\', \'shuf2\']:\n        assert \' \'.join(sents[i] for i in o[k]) == yo[k], k\n    assert order_indices([[\'a\'], [\'b\'], [\'c\', \'d\']])[\'para_lead\'] == [0, 2, 3, 1]\n    assert order_indices([[\'a\', \'b\'], [\'c\', \'d\']])[\'para_lead\'] is None\n    # K_long を素直な数え方と比べる\n    rng = np.random.default_rng(3)\n    for _ in range(20):\n        p = [int(i) for i in rng.permutation(9)]\n        pos = {v: i for i, v in enumerate(p)}\n        Kref = sum(1 for i in range(9) for j in range(i + 2, 9) if pos[j] < pos[i])\n        assert order_stats(p)[1] == Kref\n    # 当てはめ：σ ＝ 5 × lead ＋ 6 × b ＋ 0.3 × K_long を作って戻す\n    wins = []\n    for _ in range(25):\n        wins.append([[\'x\'] * int(rng.integers(1, 5)) for _ in range(int(rng.integers(3, 6)))])\n    LP = {}\n    for m in [\'m1\', \'m2\']:\n        lp = np.full((25, 3, 7), np.nan)\n        for j, w in enumerate(wins):\n            oi = order_indices(w)\n            for ci in range(3):\n                lp[j, ci, 0] = -500.0\n                for ki, k in enumerate(T_ORDERS[1:], 1):\n                    if oi[k] is None:\n                        continue\n                    b, K, ld = order_stats(oi[k])\n                    lead_scale = {0: 1.0, 1: 0.4, 2: 1.0}[ci]                # match だけ書き出しの損が小さい\n                    lp[j, ci, ki] = -500.0 - (5.0 * lead_scale * ld + 6.0 * b + 0.3 * K)\n        LP[m] = lp\n    r = analyze_title(LP, wins)\n    f = r[\'models\'][\'m1\'][\'fit_lead_adj_long\']\n    assert abs(f[\'A\'][0] - 5) < 1e-6 and abs(f[\'c_adj\'][0] - 6) < 1e-6 and abs(f[\'c_long\'][0] - 0.3) < 1e-6, f\n    ns_ = np.array([sum(len(p) for p in w) for w in wins])\n    # L は「最初の文を先頭からどかした損」：A ＋ 最初の文と離れた文の組（文の数 − 2 組）の分\n    assert abs(r[\'models\'][\'m1\'][\'lead_cost\'][\'none\'][0] - (5 + 0.3 * (ns_ - 2).mean())) < 1e-6\n    assert abs(r[\'models\'][\'m1\'][\'lead_cost_none_minus_match\'][0] - 3) < 1e-6\n    assert r[\'R1\'] == \'合うタイトルで書き出しの損が減る\' and r[\'R4\'] == \'離れたつながりがある\', (r[\'R1\'], r[\'R4\'])\n    assert r[\'R2\'] == \'無関係なタイトルは書き出しの損を変えない\', r[\'R2\']\n    assert ci_word([(1, -1, 2), (1, 0.5, 2)], \'P\', \'Z\', \'N\') == \'はっきりしない\'\n    # body_logprob を素直な計算と比べる（小さなランダムのモデル）\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    pre, text = \'Title: Something here\\nArticle:\\n\', \'One sentence. Another one here.\'\n    lp1, n1, np1 = body_logprob(model, tok, pre, text, valid, device)\n    ids = tok(pre, add_special_tokens=False).input_ids + tok(text, add_special_tokens=False).input_ids\n    z = model(torch.tensor([ids], device=device)).logits[0].double().masked_fill(~valid, float(\'-inf\'))\n    ls = torch.log_softmax(z, -1)\n    ref = sum(float(ls[i - 1, ids[i]]) for i in range(np1, len(ids)))\n    assert abs(lp1 - ref) < 1e-6 * max(1, abs(ref)) and n1 == len(ids) - np1\n    lp0, _, _ = body_logprob(model, tok, \'Article:\\n\', text, valid, device)\n    lpy, _ = seq_logprob(model, tok, text, valid, device)            # 探索 Y の関数と同じ値になるか\n    assert abs(lp0 - lpy) < 1e-9\n    p = prefixes([\'A\', \'B\', \'C\'], 2)\n    assert p[\'match\'] == \'Title: C\\nArticle:\\n\' and p[\'mismatch\'] == \'Title: A\\nArticle:\\n\'\n    return \'OK\'\n'
assert hashlib.sha256(SRC22.encode()).hexdigest() == 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53'
assert hashlib.sha256(SRCXY.encode()).hexdigest() == '5c5750e76358b86a1840e6894f9cd86c377e6be1f0b9d4104998b5144a1faaea'
assert hashlib.sha256(SRCT.encode()).hexdigest() == '62cd36ef91212dba93ca2a8820d39edf51562ad25a93b6ccc0f431b7e3c9ea52'
exec(SRC22)
exec(SRCXY)
exec(SRCT)
print('コードを置きました（', _p, 'から 2 つ取り出しました）')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ 4 モデルで 3 条件 × 7 並べ替え → 保存（結果は表示しない）
import json
try:
    print('自己テスト：', self_test('cuda'), self_test_xy('cuda'), self_test_title('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_explore', exist_ok=True)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    r = run_title(docs, '/content/drive/MyDrive/logit-thermo/A2_explore', 'cuda', notify=lambda m: notify('✅ ' + m, tags='white_check_mark'))
    r['a2_explore_title_sha256'] = '62cd36ef91212dba93ca2a8820d39edf51562ad25a93b6ccc0f431b7e3c9ea52'
    json.dump(r, open('/content/drive/MyDrive/logit-thermo/A2_explore' + '/title_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 Y の続き 2で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：タイトルと離れたつながりの計算を A2_explore/title_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
